In [1]:
import matplotlib.pyplot as plt
import xarray as xr
from glob import glob
import pandas as pd
import numpy as np

import MLD_ as m
from scipy import interpolate
import statsmodels.api as sm

import datetime as dt
import warnings

from scipy import signal
import importlib


Object `interpolate.interp1d` not found.


In [5]:
def regrid_2_SigTheta(ds, z_orig, var, step):
    """
    Interpolate Raw BGC float profiles to constant density grid with levels set by step kwd. 
    data should be first visually checked. Before interpolation check for density inversions. 
    This needs to be done because density does not always increase with pressure even in delayed mode data (T, S that lead to 
    small sigma theta inversions (<.05kg/m3) are flagged good http://www.argodatamgt.org/content/download/348/2691/file/argo_standard_tests.pdf)
    check each profile and when spikes are found, save out profile with and without median filter. 
    check if the density inversion/spike remains, if so set profile to nan. 
    Otherwise, when regridding oxygen to constant density surfaces, will mean values from greater pressures
    will be incorrectly assigned to shallower density surface. 
    Apply a 5pt median filter to all vertical profiles before interpolation
    """
    density_range=np.arange(ds.Sigma_theta.min().values, ds.Sigma_theta.max().values, step)
    density_grid=np.zeros((ds.JULD.shape[0], density_range.shape[0]))
    new_var=np.zeros(density_grid.shape)
        
    for i in range(ds.JULD.shape[0]):
        x=ds.Sigma_theta.isel(JULD=i).values
        if var==z_orig: y=ds[var].values
        else: y=ds[var].isel(JULD=i).values
        if np.isnan(y).all()==True:
            continue
        elif ds.Sigma_theta.isel(JULD=i).diff(dim=z_orig).any()>0:
            #print("Density not always increasing with pressure")        
            x_allincreasing_nonans=x[~(np.isnan(y))]#[::-1]
            x_filtered=signal.medfilt(x_allincreasing_nonans, 5)
            y_allincreasing_nonans=y[~(np.isnan(y))]#[::-1]
            y_filtered=signal.medfilt(y_allincreasing_nonans, 5)
            #filter removes small inversion, but if dens inversion is near step (0.005), will create depth inversions w/ interpolation
            if np.diff(x_filtered).max()<-0.005:
                print(x_filtered, "sigma drop na")
                plt.plot(np.diff(x_filtered))
                plt.show()
                print("!! dens inversion <-0.005 after filter,-setting profile to NAN")
                #print(x_filtered)
                new_var[i]=np.full(len(density_range), np.nan)
                continue

        else:
            x_allincreasing_nonans=x[~(np.isnan(y))]#[::-1]
            x_filtered=signal.medfilt(x_allincreasing_nonans, 5)
            y_allincreasing_nonans=y[~(np.isnan(y))]#[::-1]
            y_filtered=signal.medfilt(y_allincreasing_nonans, 5)

        
        interp_func = interpolate.interp1d(x_filtered, y_filtered, bounds_error=False, fill_value="Nan")
        new_var[i] = interp_func(density_range)

    da_new_var=xr.DataArray(new_var, dims=(["JULD", "Sigma_theta"]), coords={"JULD":ds.JULD.values, "Sigma_theta":density_range})
    return da_new_var
    

In [8]:
variables = ['Salinity','Temperature','Oxygen','Nitrate','OxygenSat','Chl_a', 'b_bp700','Snorm_Nitrate',
    'MLD','Float_ID','Pressure','Depth','Sigma_theta']


In [43]:
### NOTE kz is 10**-5 in the analysis
def Prep_data_may25update(ds, P_slice):
   
    print(ds.Oxygen.size)
    #print(ds.Oxygen.dropna(dim="N_LEVELS").count())
    # Check if over 85% of the DataArray is np.nan
    nan_percentage = np.sum(np.isnan(ds.Oxygen.values)) / ds.Oxygen.size
    threshold_percentage = 0.85
    if nan_percentage > threshold_percentage or len(ds.Oxygen.JULD.values)<3:
        print(f"Over {threshold_percentage * 100}% of the DataArray is np.nan.")
        print(ds.Float_ID.values)
        ds_Pgrid=xr.Dataset()
        return ds_Pgrid
    else:
        ds=calc_mld(ds, 10,.03)
        #ds=ds.resample(JULD="1D").interpolate()
        #Depth difference between levels in m
        Z_diff=ds.Depth.diff(dim="N_LEVELS")
        centered_Depth=ds.Depth.isel(N_LEVELS=slice(1, None))-Z_diff/2
        centered_Depth_dz=centered_Depth.diff(dim="N_LEVELS")
        Oz_diff=ds.Oxygen.diff(dim="N_LEVELS", n=2)
        #Timestep in seconds
        t_step_sec=ds.JULD.diff(dim="JULD")/ (np.timedelta64(1, 's'))
        #print(t_step_sec)
        F_diff=(10**-5*(Oz_diff/centered_Depth_dz).isel(JULD=slice(1, None))*(t_step_sec))
        #print(F_diff.max(), "FDIFF MAX")
        #trim upper level and first profile of the dataset so same shape as F_diff
        trimmed_DS=ds.isel(JULD=slice(1,None), N_LEVELS=slice(1,-1))
        #print(trimmed_DS.Oxygen)
        #assign Fdiff
        #trimmed_DS["bioOxygen"]=trimmed_DS.Oxygen-F_diff
        trimmed_DS["F_diff"]=xr.DataArray(F_diff, coords={"JULD":F_diff.JULD})

        #1-dbar pressure interp
        ds_Pgrid=dsvars_togrid_v2(trimmed_DS ,1, Pressure=True)
        #ds_Pgrid["bioOxygen"]=ds_Pgrid.Oxygen-ds_Pgrid.F_diff
        #changing so that F-diff is subtracted from the profiling following the one which it is calculated from, 
        #ie, now subtracting the change in concetration between profiles due to diffusion
        new_bioOx=(ds_Pgrid.Oxygen.isel(JULD=slice(1,None)).values-ds_Pgrid.F_diff.isel(JULD=slice(0, -1)).values)
        new_bioOx_da=xr.DataArray(new_bioOx, coords=ds_Pgrid.Oxygen.isel(JULD=slice(1,None)).coords)
        ds_Pgrid["bioOxygen"]=new_bioOx_da
        ds_Pgrid=ds_Pgrid.assign_coords({"Pressure":ds_Pgrid.Pressure})
        ds_Pgrid=ds_Pgrid.sel(Pressure=P_slice)

        return ds_Pgrid


In [49]:
def regridDS(ds_in, z_orig, step):
    print(step, "step")
    ds_out=xr.Dataset()
    ds_out["Float_ID"]=ds_in.Float_ID
    ds_out["Depth"]=regrid_2_SigTheta(ds_in,  z_orig,"Depth", step)
    ds_out["Oxygen"]=regrid_2_SigTheta(ds_in,  z_orig,"Oxygen", step)
    ds_out["bioOxygen"]=regrid_2_SigTheta(ds_in,  z_orig,"bioOxygen", step)
    ds_out["Salinity"]=regrid_2_SigTheta(ds_in, z_orig,"Salinity", step)
    if z_orig =="N_LEVELS":
        ds_out["MLD"]=ds_in.MLD.isel(N_LEVELS=0)
    else:
        ds_out["MLD"]=xr.DataArray(ds_in.MLD.values, dims=ds_in.JULD.dims)
        ds_out["Pressure"]=regrid_2_SigTheta(ds_in, "Pressure", "Pressure",step)
    ds_out["F_diff"]=regrid_2_SigTheta(ds_in,  z_orig,"F_diff", step)
    ds_out["Lat"]=xr.DataArray(ds_in.Lat.values, dims=ds_in.JULD.dims)
    ds_out["Lon"]=xr.DataArray(ds_in.Lon.values, dims=ds_in.JULD.dims)
    return ds_out